# B-Scan Migration Playground — Subwavelength PSF TimeLapse Analysis

**Purpose:** Quantify the lateral and vertical movement detection limit of two subwavelength PEC point scatterers
using Point Spread Functions from a gprMax **zero-offset B-scan** simulation.

## Geometry

## Migration methods (all post-stack / zero-offset)
| Method | Domain | Velocity use |
|--------|--------|---------------|

## Sensitivity sweep
The right scatterer is fixed, the left scatterer moves w.r.t. a baseline study to create the timelapsed study. The right scatterer moves laterally or vertically by a fraction of the wavelength in the medium (2 lambda to 1/32 lambda). After migration the sensitivity of is determined if this movement can be detected. Both migration before and after taking the timelapse difference is tested.

In [ ]:
import os, time as _time
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')

import os
import numpy as np
import pyvista as pv
import matplotlib.pyplot as plt
from gprMax.gprMax import api
from tools.outputfiles_merge import merge_files
from tools.plot_Bscan import get_output_data, mpl_plot

In [ ]:
# ── Standard figure auto-save (protocol: .wiki/FIGURES_PROTOCOL.md) ─────────
import sys, pathlib
if str(pathlib.Path.cwd()) not in sys.path:
    sys.path.insert(0, str(pathlib.Path.cwd()))
from helper_functions.figures import setup_autosave
setup_autosave(study="TimeLapse_Study", prefix="TL_")


In [ ]:
from helper_functions.visualisation import (
    plot_domain_geometry, plot_bscan_grid, plot_migrated_grid, plot_wavefield_grid,
    plot_method_comparison_grid, plot_psf_grid, plot_spectrum_grid, plot_trace_comparison
)


In [ ]:
# Medium
eps_r      = 3.15
v_ice      = 0.299792458 / np.sqrt(eps_r)   # m/ns ≈ 0.16892
f_c_GHz    = 1.5                              # centre frequency [GHz]
wavelength = v_ice / f_c_GHz                  # m ≈ 0.1126
v_mig      = v_ice / 2                        # exploding-reflector half-velocity [m/ns]
t0_ns      = np.sqrt(2) / f_c_GHz            # Ricker peak delay [ns] ≈ 0.943

# Survey geometry
domain_x   = 4.0
n_traces   = 380
trace_step = 0.01   # m
rx_offset  = 0.1    # m (source-receiver offset in .in file)
x_traces   = (0.1 + rx_offset / 2) + np.arange(n_traces) * trace_step   # midpoints [m]

# Scatterer geometry (shared across all datasets)
x_centre        = domain_x / 2
y_surface       = 0.9                          # air-ice interface [m from bottom]
y_scatterer     = y_surface - wavelength * 6  # cylinder centre [m from bottom]
z_scatterer     = y_surface - y_scatterer     # depth below surface [m] ≈ 0.676
radius_scat     = wavelength / 40             # radius of cylinder (point scatterer approximation) [m] ≈ 0.0028
z_top           = z_scatterer - radius_scat   # depth to cylinder top (first reflection)

separations = np.array([2, 1, 0.5, 0.25, 0.125, 0.0625, 0.03125]) * wavelength
x_s1_all    = np.round(x_centre - wavelength * 3 + separations, 3)
x_s2_all    = np.round(x_centre + wavelength * 3 - separations, 3)
labels      = ['Baseline', '2λ', '1λ', '½λ', '¼λ', '⅛λ', '¹⁄₁₆λ', '¹⁄₃₂λ']

# Migration depth grid
z_img = np.linspace(0.0, 0.8, 160)

# Paths
STUDY_ROOT = Path(r'C:\Users\Administrator\OneDrive\Thesis\TimeLapse_Notebooks\timelapse_study')

print(f'λ_ice       = {wavelength*1e3:.1f} mm')
print(f'v_ice       = {v_ice:.5f} m/ns,  v_mig = {v_mig:.5f} m/ns')
print(f't0_ns       = {t0_ns:.3f} ns')
print(f'z_scatterer = {z_scatterer:.3f} m  (centre),  z_top = {z_top:.4f} m')
print(f'x_traces    : {x_traces[0]:.3f} → {x_traces[-1]:.3f} m  ({n_traces} traces)')

# Importing Noise

In [ ]:
import pickle
from scipy import stats
with open('laplace_noise_model.pkl', 'rb') as f:
    noise_model = pickle.load(f)

with open('laplace_noise_model_pre_gain.pkl', 'rb') as f:
    noise_model_pre_gain = pickle.load(f)

# Input File Creation

## Calculate Geometry Factors

In [ ]:
separation = np.array([0, 2, 1, 1/2, 1/4, 1/8, 1/16, 1/32]) * wavelength  # m

x_centre = domain_x / 2
x_scatterer_1 = np.round(x_centre - wavelength * 3 + separation, 3) # 8 locations, scatterer that moves
x_scatterer_2 = np.round(x_centre + wavelength * 3, 3) # 8 locations, scatterer that is fixed

# check if discretisation is sufficient for CFL
dx_required = (v_ice / 4.5) / 10 # 4 GHz is the highest frequency component in the Ricker wavelet
if 0.001 < dx_required:
    print(f"Discretisation is sufficiently small")
if radius_scat < dx_required:
    print(f"Scatterer radius is smaller than discretisation")

# Load Cached Results (optional shortcut)

In [ ]:
USE_CACHED_RESULTS = True

if USE_CACHED_RESULTS:
    # Loads the .npz archives written by the "Saving the Data" / "Save Migrated
    # Results as np arrays dictionary" / "Save Unmigrated Noisy B-scans" /
    # "Save Migrated Results (Noisy, non-difference)" cells further down this
    # notebook, and reconstructs the variables the plotting cells consume —
    # so you can skip the gprMax file-writing, raw-data-loading, noise
    # injection, and Kirchhoff/Gazdag migration cells entirely.
    _static   = np.load(STUDY_ROOT / 'static_results.npz')
    _static_n = np.load(STUDY_ROOT / 'static_results_noisy.npz')
    _mig      = np.load(STUDY_ROOT / 'migrated_results.npz')
    _mig_diff = np.load(STUDY_ROOT / 'difference_migrated_results.npz')
    _mig_n    = np.load(STUDY_ROOT / 'migrated_results_noisy.npz')

    # ── Re-derive labels_all / labels exactly as the "Loading in the data and
    # removing direct wave" cell does further down (labels_all keeps
    # 'Baseline', labels drops it) — every migration/plotting cell below
    # assumes this split has already happened. Guarded so re-running this
    # cell doesn't re-slice `labels` a second time.
    if 'labels_all' not in dir():
        labels_all = list(labels)   # 8 entries: Baseline + 7 shifts
        labels     = labels[1:]     # 7 entries: shifts only (matches diff archives)

    # ── Raw (background-subtracted) B-scans, clean and noisy ────────────────────
    outputs_static = list(_static['data_static'])          # 8 × (n_t, n_traces)
    data_static     = list(zip(labels_all, outputs_static))

    outputs_noisy  = list(_static_n['data_static'])         # 8 × (n_t, n_traces)
    data_noisy      = list(zip(labels_all, outputs_noisy))
    NOISE_LEVEL     = float(_static_n['noise_level'])

    time_ns = _static['time_ns']
    dt      = float(_static['dt'])
    dt_ns   = dt * 1e9
    n_t     = outputs_static[0].shape[0]

    # ── Migrated results (Kirchhoff / Gazdag), clean and noisy ──────────────────
    migrated          = dict(zip(_mig['scenarios'],   _mig['kirchhoff']))
    migrated_gz       = dict(zip(_mig['scenarios'],   _mig['gazdag']))
    migrated_noisy    = dict(zip(_mig_n['scenarios'], _mig_n['kirchhoff']))
    migrated_gz_noisy = dict(zip(_mig_n['scenarios'], _mig_n['gazdag']))

    # ── TimeLapse differences (migrated[label] − migrated['Baseline']); keys
    # are `labels` (7 shift scenarios, Baseline excluded) — matches the
    # diff archive's scenario axis exactly.
    migrated_diff    = dict(zip(_mig_diff['scenarios'], _mig_diff['kirchhoff_diff']))
    migrated_gz_diff = dict(zip(_mig_diff['scenarios'], _mig_diff['gazdag_diff']))

    # ── Plot extents / constants that downstream cells expect in scope ─────────
    extent_mig   = [x_traces[0], x_traces[-1], z_img[-1], z_img[0]]
    extent_bscan = [x_traces[0], x_traces[-1], time_ns[-1], 0]
    ANGLE_AP     = 40   # must match the Kirchhoff-migration cell's value; only used for title text here

    # ── Sanity-check the cached geometry against what the cheap cells above
    # (Medium / CFL check) already computed, to catch a stale STUDY_ROOT early ─
    assert np.allclose(_mig['x_s1'], x_scatterer_1), 'cached x_s1 disagrees with x_scatterer_1'
    assert np.isclose(float(_mig['x_s2']), x_scatterer_2), 'cached x_s2 disagrees with x_scatterer_2'
    assert np.isclose(float(_mig['z_top']), z_top), 'cached z_top disagrees with z_top'

    print('Loaded cached results from:', STUDY_ROOT)
    print(f'  static_results.npz              -> data_static / outputs_static     {_static["data_static"].shape}')
    print(f'  static_results_noisy.npz        -> data_noisy / outputs_noisy       {_static_n["data_static"].shape}  (NOISE_LEVEL={NOISE_LEVEL})')
    print(f'  migrated_results.npz            -> migrated, migrated_gz            {_mig["kirchhoff"].shape}  scenarios={list(_mig["scenarios"])}')
    print(f'  difference_migrated_results.npz -> migrated_diff, migrated_gz_diff  {_mig_diff["kirchhoff_diff"].shape}  scenarios={list(_mig_diff["scenarios"])}')
    print(f'  migrated_results_noisy.npz      -> migrated_noisy, migrated_gz_noisy {_mig_n["kirchhoff"].shape}')
    print()
    print('NOT covered by these caches (re-run the corresponding cells if you need them):')
    print('  - data_pre / output_background / raw_outputs — the pre-background-subtraction')
    print('    "Visualisation" cell (Background/Baseline/shift B-scans before subtraction)')
    print('    is not cached; only background-subtracted data_static/data_noisy are.')
    print('  - Back-propagation raw wavefield snapshots (focus_frames, focus_frames_noisy,')
    print('    bp_ez_diff) — the "Back-Propagation Migration" sections (clean & noisy) need')
    print('    the .vti snapshot files re-read via pyvista. A migration-grid-reprojected')
    print("    version of the back-prop images does exist inside migrated_results*.npz")
    print("    ('backprop' key) and difference_migrated_results.npz ('backprop_diff' key),")
    print('    but is intentionally NOT exposed as a variable here since it is the wrong')
    print('    domain/resolution for the plot_wavefield_grid() calls that consume')
    print('    focus_frames/bp_ez_diff (those plot the full x-y gprMax grid, not the')
    print('    migration image grid).')
    print('  - "Analysis" section (comparison grid + PSF plots) — builds `imgs` from')
    print('    bp_ez_diff (raw domain, see above) plus the LSM (CGLS) column, which is')
    print('    already vestigial/dead in this notebook (m_cgls/m_adj are never defined)')
    print('    independent of this shortcut cell.')


## Background model (no scatterers)

In [ ]:
%%writefile timelapse_study/background/background.in

#title: GPR Timelapse Study - background
#domain: 4.000 1.000 0.001
#dx_dy_dz: 0.001 0.001 0.001
#time_window: 20e-9
#pml_cells: 10 10 0 10 10 0

#material: 3.15 1e-6 1.0 0 ice

// Subsurface background model
#box: 0 0 0 4.000 0.900 0.001 ice

#python:
from gprMax.input_cmd_funcs import *

# current_model_run starts at 1 and goes up to the -n value
iteration = current_model_run - 1  # Offset to start at 0

# Calculate current x-position
curr_x = 0.100 + (iteration * 0.01)

# Place exactly ONE source and ONE receiver
waveform('ricker', 1.0, 1.5e9, 'my_ricker')
hertzian_dipole('z', curr_x, 0.900, 0, 'my_ricker')
rx(curr_x + 0.1, 0.900, 0)
#end_python:



#geometry_view: 0 0 0 4.000 1.000 0.001 0.001 0.001 0.001 background n
#messages: y

# Baseline (static scatterers)

In [ ]:
%%writefile timelapse_study/baseline/baseline.in

#title: GPR Timelapse Study - baseline
#domain: 4.000 1.000 0.001
#dx_dy_dz: 0.001 0.001 0.001
#time_window: 20e-9
#pml_cells: 10 10 0 10 10 0

#material: 3.15 1e-6 1.0 0 ice

// Subsurface background model
#box: 0 0 0 4.000 0.900 0.001 ice

// Two point scatterers (PEC)
#cylinder: 1.662 0.224 0 1.662 0.224 0.001 0.0028 pec
#cylinder: 2.338 0.224 0 2.338 0.224 0.001 0.0028 pec

#python:
from gprMax.input_cmd_funcs import *

# current_model_run starts at 1 and goes up to the -n value
iteration = current_model_run - 1  # Offset to start at 0

# Calculate current x-position
curr_x = 0.100 + (iteration * 0.01)

# Place exactly ONE source and ONE receiver
waveform('ricker', 1.0, 1.5e9, 'my_ricker')
hertzian_dipole('z', curr_x, 0.900, 0, 'my_ricker')
rx(curr_x + 0.1, 0.900, 0)
#end_python:



#geometry_view: 0 0 0 4.000 1.000 0.001 0.001 0.001 0.001 baseline n
#messages: y

## 2 $\lambda$ lateral shift

In [ ]:
%%writefile timelapse_study/shift_2lambda/resolution_2lambda.in

#title: GPR Timelapse Study - 2 wavelength shift
#domain: 4.000 1.000 0.001
#dx_dy_dz: 0.001 0.001 0.001
#time_window: 20e-9
#pml_cells: 10 10 0 10 10 0

#material: 3.15 1e-6 1.0 0 ice

// Subsurface background model
#box: 0 0 0 4.000 0.900 0.001 ice

// Two point scatterers (PEC)
#cylinder: 1.887 0.224 0 1.887 0.224 0.001 0.0028 pec
#cylinder: 2.338 0.224 0 2.338 0.224 0.001 0.0028 pec

#python:
from gprMax.input_cmd_funcs import *

# current_model_run starts at 1 and goes up to the -n value
iteration = current_model_run - 1  # Offset to start at 0

# Calculate current x-position
curr_x = 0.100 + (iteration * 0.01)

# Place exactly ONE source and ONE receiver
waveform('ricker', 1.0, 1.5e9, 'my_ricker')
hertzian_dipole('z', curr_x, 0.900, 0, 'my_ricker')
rx(curr_x + 0.1, 0.900, 0)
#end_python:



#geometry_view: 0 0 0 4.000 1.000 0.001 0.001 0.001 0.001 two_point_scatterers_2_lambda n
#messages: y

## 1 $\lambda$ lateral shift

In [ ]:
%%writefile timelapse_study/shift_1lambda/resolution_1lambda.in

#title: GPR Timelapse Study - 1 wavelength shift
#domain: 4.000 1.000 0.001
#dx_dy_dz: 0.001 0.001 0.001
#time_window: 20e-9
#pml_cells: 10 10 0 10 10 0

#material: 3.15 1e-6 1.0 0 ice

// Subsurface background model
#box: 0 0 0 4.000 0.900 0.001 ice

// Two point scatterers (PEC)
#cylinder: 1.775 0.224 0 1.775 0.224 0.001 0.0028 pec
#cylinder: 2.338 0.224 0 2.338 0.224 0.001 0.0028 pec

#python:
from gprMax.input_cmd_funcs import *

# current_model_run starts at 1 and goes up to the -n value
iteration = current_model_run - 1  # Offset to start at 0

# Calculate current x-position
curr_x = 0.100 + (iteration * 0.01)

# Place exactly ONE source and ONE receiver
waveform('ricker', 1.0, 1.5e9, 'my_ricker')
hertzian_dipole('z', curr_x, 0.900, 0, 'my_ricker')
rx(curr_x + 0.1, 0.900, 0)
#end_python:



#geometry_view: 0 0 0 4.000 1.000 0.001 0.001 0.001 0.001 two_point_scatterers_1_lambda n
#messages: y

## 1/2 $\lambda$ lateral shift

In [ ]:
%%writefile timelapse_study/shift_0p5lambda/resolution_0p5lambda.in

#title: GPR Timelapse Study - 0.5 wavelength shift
#domain: 4.000 1.000 0.001
#dx_dy_dz: 0.001 0.001 0.001
#time_window: 20e-9
#pml_cells: 10 10 0 10 10 0

#material: 3.15 1e-6 1.0 0 ice

// Subsurface background model
#box: 0 0 0 4.000 0.900 0.001 ice

// Two point scatterers (PEC)
#cylinder: 1.718 0.224 0 1.718 0.224 0.001 0.0028 pec
#cylinder: 2.338 0.224 0 2.338 0.224 0.001 0.0028 pec

#python:
from gprMax.input_cmd_funcs import *

# current_model_run starts at 1 and goes up to the -n value
iteration = current_model_run - 1  # Offset to start at 0

# Calculate current x-position
curr_x = 0.100 + (iteration * 0.01)

# Place exactly ONE source and ONE receiver
waveform('ricker', 1.0, 1.5e9, 'my_ricker')
hertzian_dipole('z', curr_x, 0.900, 0, 'my_ricker')
rx(curr_x + 0.1, 0.900, 0)
#end_python:


#geometry_view: 0 0 0 4.000 1.000 0.001 0.001 0.001 0.001 two_point_scatterers_0p5_lambda n
#messages: y

## 1/4 $\lambda$ lateral shift

In [ ]:
%%writefile timelapse_study/shift_0p25lambda/resolution_0p25lambda.in

#title: GPR Timelapse Study - 0.25 wavelength shift
#domain: 4.000 1.000 0.001
#dx_dy_dz: 0.001 0.001 0.001
#time_window: 20e-9
#pml_cells: 10 10 0 10 10 0

#material: 3.15 1e-6 1.0 0 ice

// Subsurface background model
#box: 0 0 0 4.000 0.900 0.001 ice

// Two point scatterers (PEC)
#cylinder: 1.690 0.224 0 1.690 0.224 0.001 0.0028 pec
#cylinder: 2.338 0.224 0 2.338 0.224 0.001 0.0028 pec

#python:
from gprMax.input_cmd_funcs import *

# current_model_run starts at 1 and goes up to the -n value
iteration = current_model_run - 1  # Offset to start at 0

# Calculate current x-position
curr_x = 0.100 + (iteration * 0.01)

# Place exactly ONE source and ONE receiver
waveform('ricker', 1.0, 1.5e9, 'my_ricker')
hertzian_dipole('z', curr_x, 0.900, 0, 'my_ricker')
rx(curr_x + 0.1, 0.900, 0)
#end_python:



#geometry_view: 0 0 0 4.000 1.000 0.001 0.001 0.001 0.001 two_point_scatterers_0p25_lambda n
#messages: y

## 1/8 $\lambda$ lateral shift

In [ ]:
%%writefile timelapse_study/shift_0p125lambda/resolution_0p125lambda.in

#title: GPR Timelapse Study - 0.125 wavelength shift
#domain: 4.000 1.000 0.001
#dx_dy_dz: 0.001 0.001 0.001
#time_window: 20e-9
#pml_cells: 10 10 0 10 10 0

#material: 3.15 1e-6 1.0 0 ice

// Subsurface background model
#box: 0 0 0 4.000 0.900 0.001 ice

// Two point scatterers (PEC)
#cylinder: 1.676 0.224 0 1.676 0.224 0.001 0.0028 pec
#cylinder: 2.338 0.224 0 2.338 0.224 0.001 0.0028 pec

#python:
from gprMax.input_cmd_funcs import *

# current_model_run starts at 1 and goes up to the -n value
iteration = current_model_run - 1  # Offset to start at 0

# Calculate current x-position
curr_x = 0.100 + (iteration * 0.01)

# Place exactly ONE source and ONE receiver
waveform('ricker', 1.0, 1.5e9, 'my_ricker')
hertzian_dipole('z', curr_x, 0.900, 0, 'my_ricker')
rx(curr_x + 0.1, 0.900, 0)
#end_python:



#geometry_view: 0 0 0 4.000 1.000 0.001 0.001 0.001 0.001 two_point_scatterers_0p125_lambda n
#messages: y

## 1/16 $\lambda$ lateral shift

In [ ]:
%%writefile timelapse_study/shift_0p0625lambda/resolution_0p0625lambda.in

#title: GPR Timelapse Study - 0.0625 wavelength shift
#domain: 4.000 1.000 0.001
#dx_dy_dz: 0.001 0.001 0.001
#time_window: 20e-9
#pml_cells: 10 10 0 10 10 0

#material: 3.15 1e-6 1.0 0 ice

// Subsurface background model
#box: 0 0 0 4.000 0.900 0.001 ice

// Two point scatterers (PEC)
#cylinder: 1.669 0.224 0 1.669 0.224 0.001 0.0028 pec
#cylinder: 2.338 0.224 0 2.338 0.224 0.001 0.0028 pec

#python:
from gprMax.input_cmd_funcs import *

# current_model_run starts at 1 and goes up to the -n value
iteration = current_model_run - 1  # Offset to start at 0

# Calculate current x-position
curr_x = 0.100 + (iteration * 0.01)

# Place exactly ONE source and ONE receiver
waveform('ricker', 1.0, 1.5e9, 'my_ricker')
hertzian_dipole('z', curr_x, 0.900, 0, 'my_ricker')
rx(curr_x + 0.1, 0.900, 0)
#end_python:



#geometry_view: 0 0 0 4.000 1.000 0.001 0.001 0.001 0.001 two_point_scatterers_0p0625_lambda n
#messages: y

## 1/32 $\lambda$ lateral shift

In [ ]:
%%writefile timelapse_study/shift_0p03125lambda/resolution_0p03125lambda.in

#title: GPR Timelapse Study - 0.0625 wavelength shift
#domain: 4.000 1.000 0.001
#dx_dy_dz: 0.001 0.001 0.001
#time_window: 20e-9
#pml_cells: 10 10 0 10 10 0

#material: 3.15 1e-6 1.0 0 ice

// Subsurface background model
#box: 0 0 0 4.000 0.900 0.001 ice

// Two point scatterers (PEC)
#cylinder: 1.666 0.224 0 1.666 0.224 0.001 0.0028 pec
#cylinder: 2.338 0.224 0 2.338 0.224 0.001 0.0028 pec

#python:
from gprMax.input_cmd_funcs import *

# current_model_run starts at 1 and goes up to the -n value
iteration = current_model_run - 1  # Offset to start at 0

# Calculate current x-position
curr_x = 0.100 + (iteration * 0.01)

# Place exactly ONE source and ONE receiver
waveform('ricker', 1.0, 1.5e9, 'my_ricker')
hertzian_dipole('z', curr_x, 0.900, 0, 'my_ricker')
rx(curr_x + 0.1, 0.900, 0)
#end_python:



#geometry_view: 0 0 0 4.000 1.000 0.001 0.001 0.001 0.001 two_point_scatterers_0p03125_lambda n
#messages: y

# Visualising Model Geometry

In [ ]:
from matplotlib.patches import Rectangle, Circle
import matplotlib.pyplot as plt

# ── Geometry constants (from input files) ─────────────────────────────────────
domain_y = 1.0
dx_grid  = 0.001
pml_t    = 10 * dx_grid   # = 0.010 m
air_h    = domain_y - y_surface   # = 0.1 m
rx_off   = 0.1            # m  Tx-Rx offset

# Depth coordinate (0 = ice surface, positive = deeper)
d_top  = -air_h          # −0.1 m  top of domain
d_scat =  z_scatterer    #  0.676 m cylinder centre

# Scenario labels and scatterer x-positions  (8 used: Baseline + 7 shifts)
_labels = ['Baseline', '2λ', '1λ', '½λ', '¼λ', '⅛λ', '¹⁄₁₆λ', '¹⁄₃₂λ']
_x_sc1  = x_scatterer_1[:8]           # moving scatterer  (array, 8 values)
_x_sc2  = float(x_scatterer_2)        # fixed scatterer   (scalar)

# ── Figure 1: Full domain cross-section ──────────────────────────────────────
fig1, ax1 = plot_domain_geometry(
    domain_x, domain_y, y_surface, pml_t,
    x_src=0.100 + np.arange(n_traces) * trace_step, rx_offset=rx_off,
    eps_r=eps_r,
)

# Scatterers at baseline positions (radius x4 for visibility)
r_vis = 4 * radius_scat
ax1.add_patch(Circle((_x_sc1[0], d_scat), r_vis,
                     facecolor='tomato',    edgecolor='#800', lw=0.8, zorder=5))
ax1.add_patch(Circle((_x_sc2,    d_scat), r_vis,
                     facecolor='royalblue', edgecolor='#004', lw=0.8, zorder=5))

# Baseline separation annotation
mid_x = (_x_sc1[0] + _x_sc2) / 2
ax1.annotate('', xy=(_x_sc2, d_scat - 0.04), xytext=(_x_sc1[0], d_scat - 0.04),
             arrowprops=dict(arrowstyle='<->', color='k', lw=0.9))
ax1.text(mid_x, d_scat - 0.055,
         f'baseline sep = {(_x_sc2 - _x_sc1[0])*1e3:.0f} mm = 6λ',
         ha='center', va='top', fontsize=8)

# s1/s2 labels
ax1.text(_x_sc1[0], d_scat + 0.025, 's1\n(moving)', ha='center', va='bottom', fontsize=7.5, color='#800')
ax1.text(_x_sc2,    d_scat + 0.025, 's2\n(fixed)',  ha='center', va='bottom', fontsize=7.5, color='#004')

ax1.set_xlim(0, domain_x)
ax1.set_ylim(y_surface, d_top)
ax1.set_xlabel('x [m]', fontsize=11)
ax1.set_ylabel('Depth [m]', fontsize=11)
ax1.set_title(
    f'TimeLapse Study — Model Geometry  '
    f'(domain {domain_x:.1f}×{domain_y:.1f} m, Δx = {dx_grid*1e3:.0f} mm, '
    f'PML = {pml_t*1e3:.0f} mm, f_c = {f_c_GHz} GHz, λ = {wavelength*1e3:.1f} mm)  '
    f'tomato = moving s1   blue = fixed s2   (baseline positions, radius ×4)',
    fontsize=10
)
ax1.set_aspect('equal', adjustable='box')
ax1.legend(loc='upper right', fontsize=9, ncol=2)
ax1.grid(True, ls='--', alpha=0.3)
plt.tight_layout()
plt.show()

# ── Figure 2: Zoomed — moving scatterer s1 positions (all 8 scenarios) ───────
x_s1_min = float(_x_sc1.min())
x_s1_max = float(_x_sc1.max())
x_s1_cen = (x_s1_min + x_s1_max) / 2
x_s1_hw  = max((x_s1_max - x_s1_min) / 2 + 5 * radius_scat, 8 * radius_scat)

fig2, ax2 = plt.subplots(figsize=(14, 6))

ax2.add_patch(Rectangle((x_s1_cen - x_s1_hw, d_scat - x_s1_hw), 2*x_s1_hw, 2*x_s1_hw,
                         facecolor='#cce5ff', edgecolor='#aac', lw=0.5))
ax2.axvline(x_s1_min, color='grey', lw=0.7, ls=':', alpha=0.4)

cmap = plt.cm.plasma
for i, (lbl, xc) in enumerate(zip(_labels, _x_sc1)):
    col = cmap(i / (len(_labels) - 1))
    ax2.add_patch(Circle((xc, d_scat), radius_scat,
                         facecolor=col, edgecolor='black', lw=0.6, zorder=5))
    ax2.text(xc, d_scat + x_s1_hw * 0.55, lbl,
             ha='center', va='bottom', fontsize=9, rotation=45)

# Max-shift arrow
y_arrow = d_scat + x_s1_hw * 0.35
ax2.annotate('', xy=(_x_sc1[1], y_arrow), xytext=(_x_sc1[0], y_arrow),
             arrowprops=dict(arrowstyle='->', color='k', lw=0.9))
ax2.text((_x_sc1[0] + _x_sc1[1]) / 2, y_arrow,
         f'max shift\n{(_x_sc1[1]-_x_sc1[0])*1e3:.1f} mm = 2λ',
         ha='center', va='bottom', fontsize=8)

ax2.set_xlim(x_s1_cen - x_s1_hw, x_s1_cen + x_s1_hw)
ax2.set_ylim(d_scat + x_s1_hw, d_scat - x_s1_hw)
ax2.set_xlabel('x [m]', fontsize=11)
ax2.set_ylabel('Depth [m]', fontsize=11)
ax2.set_title(
    f'Moving scatterer s1 — all 8 scenarios  '
    f'(r = {radius_scat*1e3:.1f} mm, depth = {z_scatterer:.3f} m = {z_scatterer/wavelength:.0f}λ)\n'
    f'Fixed scatterer s2 at x = {_x_sc2:.3f} m (off to the right, baseline sep = 6λ)',
    fontsize=11
)
ax2.set_aspect('equal', adjustable='box')
ax2.grid(True, ls='--', alpha=0.3)
plt.tight_layout()
plt.show()

# Generating Geometry Files and Merged B-scans
Only uncomment and run when necessary

In [ ]:
# input_baseline      = r'timelapse_study/baseline/resolution_baseline.in'
# input_lambda_2      = r'timelapse_study/shift_2lambda/resolution_2lambda.in'
# input_lambda_1      = r'timelapse_study/shift_1lambda/resolution_1lambda.in'
# input_lambda_0p5    = r'timelapse_study/shift_0p5lambda/resolution_0p5lambda.in'
# input_lambda_0p25   = r'timelapse_study/shift_0p25lambda/resolution_0p25lambda.in'
# input_lambda_0p125  = r'timelapse_study/shift_0p125lambda/resolution_0p125lambda.in'
# input_lambda_0p0625 = r'timelapse_study/shift_0p0625lambda/resolution_0p0625lambda.in'
# input_lambda_0p03125 = r'timelapse_study/shift_0p03125lambda/resolution_0p03125lambda.in'

# api(input_lambda_2, n=1, geometry_only=True)
# api(input_lambda_1, n=1, geometry_only=True)
# api(input_lambda_0p5, n=1, geometry_only=True)
# api(input_lambda_0p25, n=1, geometry_only=True)
# api(input_lambda_0p125, n=1, geometry_only=True)
# api(input_lambda_0p0625, n=1, geometry_only=True)
# api(input_lambda_0p03125, n=1, geometry_only=True)


In [ ]:
# filename_background    = r'timelapse_study/background/background'
# filename_baseline      = r'timelapse_study/baseline/baseline'
# filename_lambda_2      = r'timelapse_study/shift_2lambda/resolution_2lambda'
# filename_lambda_1      = r'timelapse_study/shift_1lambda/resolution_1lambda'
# filename_lambda_0p5    = r'timelapse_study/shift_0p5lambda/resolution_0p5lambda'
# filename_lambda_0p25   = r'timelapse_study/shift_0p25lambda/resolution_0p25lambda'
# filename_lambda_0p125  = r'timelapse_study/shift_0p125lambda/resolution_0p125lambda'
# filename_lambda_0p0625 = r'timelapse_study/shift_0p0625lambda/resolution_0p0625lambda'
# filename_lambda_0p03125 = r'timelapse_study/shift_0p03125lambda/resolution_0p03125lambda'

# merge_files(filename_background, removefiles=True)
# merge_files(filename_baseline, removefiles=True)
# merge_files(filename_lambda_2, removefiles=True)
# merge_files(filename_lambda_1, removefiles=True)
# merge_files(filename_lambda_0p5, removefiles=True)
# merge_files(filename_lambda_0p25, removefiles=True)
# merge_files(filename_lambda_0p125, removefiles=True)
# merge_files(filename_lambda_0p0625, removefiles=True)
# merge_files(filename_lambda_0p03125, removefiles=True)


# Loading in the data and removing direct wave

In [ ]:
rxnumber    = 1
rxcomponent = 'Ez'

output_background, dt = get_output_data(
    'timelapse_study/background/background_merged.out', rxnumber, rxcomponent
)

output_baseline, _ = get_output_data(
    'timelapse_study/baseline/baseline_merged.out', rxnumber, rxcomponent
)

_paths = [
    'timelapse_study/baseline/baseline_merged.out',
    'timelapse_study/shift_2lambda/resolution_2lambda_merged.out',
    'timelapse_study/shift_1lambda/resolution_1lambda_merged.out',
    'timelapse_study/shift_0p5lambda/resolution_0p5lambda_merged.out',
    'timelapse_study/shift_0p25lambda/resolution_0p25lambda_merged.out',
    'timelapse_study/shift_0p125lambda/resolution_0p125lambda_merged.out',
    'timelapse_study/shift_0p0625lambda/resolution_0p0625lambda_merged.out',
    'timelapse_study/shift_0p03125lambda/resolution_0p03125lambda_merged.out',
]
raw_outputs      = [get_output_data(p, rxnumber, rxcomponent)[0] for p in _paths]
outputs_static   = [r - output_background for r in raw_outputs]      # background-subtracted

dt_ns   = dt * 1e9
n_t     = outputs_static[0].shape[0]
time_ns = np.arange(n_t) * dt_ns

# Convenience lists for the visualisation cells below
data_pre = [('Background', output_background)] + list(zip(labels, raw_outputs))
data_static = list(zip(labels, outputs_static))
labels_all = list(labels)           # all 7 labels (Baseline + 7 shifts)
labels = labels[1:]

print(f'dt = {dt_ns:.6f} ns,  n_t = {n_t},  t_max = {time_ns[-1]:.2f} ns')
print(f'Data shape (n_t × n_tr): {outputs_static[0].shape}  ({n_t} time samples × {n_traces} traces)')

# Visualisation

In [ ]:
extent_bscan = [x_traces[0], x_traces[-1], time_ns[-1], 0]

def _mark_pre(ax, i):
    if i == 0:  # Background panel has no scatterers
        return
    ax.axvline(x_scatterer_1[i - 1], color='green', linestyle='--', linewidth=1)
    ax.axvline(x_scatterer_2, color='green', linestyle='--', linewidth=1)

plot_bscan_grid(
    data_pre, x_traces, time_ns,
    title='GPR B-Scans — Background, Baseline and TimeLapsed Models',
    markers=[_mark_pre] * len(data_pre),
)
plt.show()

In [ ]:
def _mark_static(ax, i):
    ax.axvline(x_scatterer_1[i], color='green', linestyle='--', linewidth=1)
    ax.axvline(x_scatterer_2, color='green', linestyle='--', linewidth=1)

plot_bscan_grid(
    data_static, x_traces, time_ns,
    title='GPR B-Scans — Background Subtracted',
    markers=[_mark_static] * len(data_static),
)
plt.show()

Create Noisy Data

In [ ]:
# ── Apply the fitted Laplace noise model to every background-subtracted B-scan ──
# The Laplace fit (loc/scale) was estimated from the real, fully-processed data
# pipeline ('Spherical Gain' stage), whose amplitudes are ~4 orders of magnitude
# larger than these synthetic Ez B-scans. We keep the Laplace *shape* (loc=0,
# heavier tails than Gaussian) but rescale it so its std is 10% of each B-scan's
# own signal std — a light, realistic noise level rather than the raw fitted scale.
NOISE_LEVEL = 0.1   # target noise std as a fraction of each B-scan's signal std
rng = np.random.default_rng(0)

outputs_noisy = []
for raw in outputs_static:
    target_std    = NOISE_LEVEL * raw.std()
    scaled_scale  = target_std / np.sqrt(2)   # Var(Laplace) = 2 * scale**2
    synthetic_noise = stats.laplace.rvs(
        loc=noise_model_pre_gain['loc'], scale=scaled_scale,
        size=raw.shape, random_state=rng,
    )
    outputs_noisy.append(raw + synthetic_noise)

data_noisy = list(zip(labels_all, outputs_noisy))

print(f"Applied Laplace noise (shape from '{noise_model_pre_gain['stage']}' fit, "
      f"rescaled to {NOISE_LEVEL:.0%} of signal std) to {len(outputs_noisy)} B-scans")


In [ ]:
def _mark_noisy(ax, i):
    ax.axvline(x_scatterer_1[i], color='green', linestyle='--', linewidth=1)
    ax.axvline(x_scatterer_2, color='green', linestyle='--', linewidth=1)

plot_bscan_grid(
    data_noisy, x_traces, time_ns,
    title='GPR B-Scans — With Synthetic Laplace Noise (10% of signal std)',
    markers=[_mark_noisy] * len(data_noisy),
)
plt.show()

In [ ]:
# -- Frequency spectra: clean vs noisy B-scans -----------------------------------
# Row 1: amplitude spectrum (mean across traces) of each clean (background-
# subtracted) B-scan in data_static. Row 2: the same for the noisy B-scans in
# data_noisy -- compare rows to see how much broadband content the Laplace
# noise injection adds on top of the underlying Ricker wavelet bandwidth.
freqs_ghz = np.fft.rfftfreq(n_t, d=dt_ns)

def _spectrum(d):
    return np.abs(np.fft.rfft(d, axis=0, norm='forward')).mean(axis=1)

spectra_clean_noisy = [
    [(f'{label} (clean)', _spectrum(d)) for label, d in data_static],
    [(f'{label} (noisy)', _spectrum(d)) for label, d in data_noisy],
]
plot_spectrum_grid(spectra_clean_noisy, freqs_ghz, title=None, xlim=(0, 7))

# Difference spectrum: noisy - clean, shown as the mean absolute FFT difference per scenario
spectra_diff = [[
    (f'{label_c} (|noisy - clean|)', np.abs(_spectrum(d_n) - _spectrum(d_c)))
    for (label_c, d_c), (label_n, d_n) in zip(data_static, data_noisy)
]]
plot_spectrum_grid(
    spectra_diff, freqs_ghz,
    title='B-scan Frequency Spectrum Difference — Noisy vs Clean',
    row_colors=('C2',), xlim=(0, 7),
)

plt.show()

# Saving the Data

In [ ]:
# ── Save non-migrated (raw, background-subtracted) B-scan data ────────────────
# Stacks all 8 scenarios (Baseline + 7 shifts) from data_static into a single
# array and saves to static_results.npz.
# Index 0 = Baseline, indices 1-7 = shift scenarios matching labels_all[1:].

static_all = np.stack([d.astype(float) for _, d in data_static], axis=0)  # (8, n_t, n_traces)

# ── Save to disk ──────────────────────────────────────────────────────────────
save_path_static = STUDY_ROOT / "static_results.npz"

np.savez_compressed(
    save_path_static,
    data_static       = static_all,                                     # (8, n_t, n_traces)
    scenarios         = np.array(labels_all, dtype="U20"),               # (8,)  Baseline + 7 shifts
    separation_lambda = np.array([0, 2, 1, 0.5, 0.25, 0.125, 0.0625, 0.03125]),  # (8,)  0 = baseline
    x_traces          = x_traces,
    time_ns           = time_ns,
    dt                = np.float64(dt),
    x_s1              = np.array(x_scatterer_1),
    x_s2              = np.float64(x_scatterer_2),
    z_scatterer       = np.float64(z_scatterer),
    z_top             = np.float64(z_top),
)

print(f"Saved → {save_path_static}")
print(f"\nStacked array  (n_scenarios={static_all.shape[0]}, n_t={static_all.shape[1]}, n_tr={static_all.shape[2]}):")
print(f"  data_static   {static_all.shape}")
print(f"\nUsage example:")
print(f"  d = np.load(str(STUDY_ROOT / 'static_results.npz'), allow_pickle=False)")
print(f"  d['data_static'][0]   # Baseline B-scan → shape (n_t, n_tr)")
print(f"  d['data_static'][3]   # ¼λ shift B-scan → shape (n_t, n_tr)")

# Kirchhoff Migration

Delay-and-sum migration via the **PyLops zero-offset Kirchhoff operator**.  
Each dataset is preprocessed (tapering + t0 shift — see cell above) before being migrated.  
Green stars mark the true cylinder-top positions.

In [ ]:
import sys, pathlib
_here = pathlib.Path.cwd()
if str(_here) not in sys.path:
    sys.path.insert(0, str(_here))
from helper_functions.migration import (PylopsKirchoffMigration, gazdag_migration,
                                        write_backprop_files, dispersion_limited_cutoff,
                                        lowpass_filter_excitation)

## Tapering & t0 Shift

Two pre-processing steps applied to every B-scan before migration:

| Step | Purpose |
|------|---------|
| **Exponential decay taper** (`taper_decay_ns`) | Suppresses late arrivals, emphasises hyperbola apices |
| **Cosine end taper** (`taper_end_ns`) | Zeros the final portion of each trace to remove hyperbola tails |
| **t0 shift** | Removes the Ricker wavelet's built-in peak delay so that t = 0 aligns with the surface |

Adjust the parameters in the cell below and re-run to see their effect on the B-scan.

In [ ]:
# ── Taper & t0-shift parameters ───────────────────────────────────────────────
taper_end_ns   = 14.0   # cosine ramp onset [ns]: zeros hyperbola tails beyond this time
taper_decay_ns = 1.0    # exponential decay time constant [ns]: de-emphasises late arrivals
ANGLE_AP       = 40     # Kirchhoff aperture half-angle [degrees]

# ── Functions ─────────────────────────────────────────────────────────────────
def make_end_taper(n_samples, taper_samples):
    """Unity everywhere, then half-cosine 1→0 over the last `taper_samples`."""
    win = np.ones(n_samples)
    if taper_samples > 0:
        ramp = 0.5 * (1 + np.cos(np.pi * np.arange(taper_samples) / taper_samples))
        win[-taper_samples:] = ramp
    return win

def make_decay_taper(time_ns, tau_ns):
    """Exponential decay starting at 1.0 (t=0) with time constant tau_ns."""
    return np.exp(-time_ns / tau_ns)

def preprocess(data_nt_ntr, dt_ns, t0_ns, time_ns):
    """
    Prepare a B-scan for Kirchhoff migration.
    Input:  data_nt_ntr  (n_t, n_tr)  — background-subtracted B-scan
    Returns tapered (n_tr, n_t) and t0-shifted (n_tr, n_t) arrays.
    """
    bscan   = data_nt_ntr.T.copy()                              # → (n_tr, n_t)
    n_t     = bscan.shape[1]
    w_end   = make_end_taper(n_t, int(round(taper_end_ns / dt_ns)))
    w_decay = make_decay_taper(time_ns, taper_decay_ns)
    tapered = bscan * (w_end * w_decay)[np.newaxis, :]
    t0_samp = int(round(t0_ns / dt_ns))
    shifted = np.roll(tapered, -t0_samp, axis=1)
    shifted[:, -t0_samp:] = 0.0
    return tapered, shifted

# ── Visualise effect on the 2λ dataset ────────────────────────────────────────
_demo = outputs_static[1]    # (n_t, n_tr) — 2λ background-subtracted
_raw_tr, _shifted = preprocess(_demo, dt_ns, t0_ns, time_ns)
_tapered = _raw_tr   # preprocess returns tapered as first output

w_end_vis   = make_end_taper(n_t, int(round(taper_end_ns   / dt_ns)))
w_decay_vis = make_decay_taper(time_ns, taper_decay_ns)
taper_combined = w_end_vis * w_decay_vis

mid_tr = n_traces // 2   # representative central trace

# --- Row 1: single-trace before/after ---
plot_trace_comparison(
    traces=[
        [('Raw', _demo[:, mid_tr], 'b'), ('Tapered', _tapered[mid_tr], 'r')],
        [('Tapered', _tapered[mid_tr], 'r'),
         (f't0-shifted (−{t0_ns:.3f} ns)', _shifted[mid_tr], 'g')],
    ],
    time_ns=time_ns,
    title='Effect of Tapering and t0 Shift — 2λ dataset, single trace',
    taper_curve=taper_combined,
    vline=taper_end_ns,
)
plt.show()

# --- Row 2: full B-scan comparison ---
plot_bscan_grid(
    data=[
        ('Raw (background subtracted)', _demo),
        (f'Tapered  (decay τ={taper_decay_ns} ns, end from {taper_end_ns} ns)', _tapered.T),
        (f't0-shifted (−{t0_ns:.3f} ns)', _shifted.T),
    ],
    x_traces=x_traces, time_ns=time_ns, ncols=3,
    title='B-scan effect of tapering and t0 shift — 2λ dataset',
    shared_colorbar=False, vmax_headroom=0.5,
)
plt.show()

In [ ]:
# ── Kirchhoff migration on all 8 datasets ─────────────────────────────────────────
migrated = {}

for label, raw in zip(labels_all, outputs_static):
    _, shifted = preprocess(raw, dt_ns, t0_ns, time_ns)   # (n_tr, n_t), t0-shifted
    print(f'[{label}] Running Kirchhoff ...', end=' ', flush=True)
    t0 = _time.perf_counter()
    img = PylopsKirchoffMigration(
        shifted, time_ns, x_traces, v_ice, z_img,
        f0=f_c_GHz, angleaperture=ANGLE_AP
    )
    print(f'{_time.perf_counter()-t0:.1f} s  shape={img.shape}')
    migrated[label] = img


In [ ]:
# ── Plot: full extent + zoomed ────────────────────────────────────────────────────
extent_mig = [x_traces[0], x_traces[-1], z_img[-1], z_img[0]]

plot_migrated_grid(
    migrated, extent_mig, ncols=4,
    title=f'Kirchhoff Migration — All 8 Datasets  |  f_c={f_c_GHz} GHz  |  aperture={ANGLE_AP}°  |  z_top={z_top:.3f} m',
    marker_x=lambda i, label: x_scatterer_1[i],
    marker_z=z_top,
    marker_x_baseline=x_scatterer_1[0],
    zooms=[
        (None, None, ''),
        ((1.5, 2.5), (0.75, 0.55), '(zoomed)'),
    ],
)
plt.show()

In [ ]:
# ── Kirchhoff timelapse differences (migrated − migrated_baseline) ─────────────
migrated_diff = {label: migrated[label] - migrated['Baseline'] for label in labels}

plot_migrated_grid(
    migrated_diff, extent_mig, ncols=4,
    title=f'Kirchhoff Migration — TimeLapse Differences (migrated − migrated_baseline)  |  f_c={f_c_GHz} GHz',
    marker_x=lambda i, label: x_scatterer_1[i + 1],
    marker_z=z_top,
    marker_x_baseline=x_scatterer_1[0],
    zooms=[
        (None, None, ''),
        ((1.5, 2.5), (0.75, 0.55), '(zoomed)'),
    ],
)
plt.show()

# Gazdag Phase-Shift Migration

f-k domain migration via **PyLops `PhaseShift`** downward continuation.  
Uses the same preprocessed (tapered + t0-shifted) B-scans as Kirchhoff, but operates in the
frequency-wavenumber domain: each depth step phase-shifts the wavefield by `kz·dz`.

Key implementation details:
- `v_mig = v_ice / 2` (exploding-reflector half-velocity) used internally
- 100 % spatial zero-padding each side to avoid wrap-around in kx
- Evanescent bins (`|kx| > |f|/v_mig`) are zeroed **before** the depth loop to prevent smile artefacts accumulating at the t = 0 imaging condition
- Green stars mark the true cylinder-top positions

_____________
Add padding in space and time up to the next power of 2
_____________

In [ ]:
import pylops

## Run Migration on All Datasets

In [ ]:
# ── Gazdag phase-shift migration on all 8 datasets ────────────────────────────────
migrated_gz = {}

for label, raw in zip(labels_all, outputs_static):
    _, shifted = preprocess(raw, dt_ns, t0_ns, time_ns)   # (n_tr, n_t), t0-shifted
    print(f'[{label}] Running Gazdag ...', flush=True)
    t0 = _time.perf_counter()
    img = gazdag_migration(
        shifted.T,       # (n_t, n_tr) — time first
        x_traces, time_ns, z_img, v_ice
    )
    print(f'  Done in {_time.perf_counter()-t0:.1f} s  shape={img.shape}')
    migrated_gz[label] = img


In [ ]:
# ── Plot: full extent + zoomed ────────────────────────────────────────────────────
plot_migrated_grid(
    migrated_gz, extent_mig, ncols=4,
    title=f'Gazdag Phase-Shift Migration — All 7 Datasets  |  f_c={f_c_GHz} GHz  |  z_top={z_top:.3f} m',
    marker_x=lambda i, label: x_scatterer_1[i],
    marker_z=z_top,
    marker_x_baseline=x_scatterer_1[0],
    zooms=[
        (None, None, ''),
        ((1.5, 2.5), (0.75, 0.55), '(zoomed)'),
    ],
)
plt.show()

In [ ]:
# ── Gazdag timelapse differences (migrated − migrated_baseline) ──────────────────────
migrated_gz_diff = {label: migrated_gz[label] - migrated_gz['Baseline'] for label in labels}

plot_migrated_grid(
    migrated_gz_diff, extent_mig, ncols=3,
    title=f'Gazdag Migration — TimeLapse Differences (migrated − migrated_baseline)  |  f_c={f_c_GHz} GHz',
    marker_x=lambda i, label: x_scatterer_1[i + 1],
    marker_z=z_top,
    marker_x_baseline=x_scatterer_1[0],
    zooms=[
        (None, None, ''),
        ((1.5, 2.5), (0.75, 0.55), '(zoomed)'),
    ],
)
plt.show()

# LSM

# Back-Propagation Migration

Time-reversal migration: re-inject the time-reversed, normalised B-scan traces at the
original receiver positions through a **half-velocity medium** (eps_r × 4 so v → v_ice/2).

Under the exploding-reflector hypothesis every scatterer focuses simultaneously at

> **t_focus = T − t0_ns**

where T is the total simulation window. Snapshots are captured in a narrow window just
before t_focus so the collapsed wavefield image can be extracted.

| Parameter | Value |
|-----------|-------|
| Injection positions | Rx locations (Tx + 0.1 m offset) |
| Medium | eps_r × 4 → v = v_ice / 2 |
| Source stride | every `STRIDE`-th trace (tunes performance vs coverage) |

In [ ]:
labels_all = ['Baseline', '2λ', '1λ', '½λ', '¼λ', '⅛λ', '¹⁄₁₆λ', '¹⁄₃₂λ']

In [ ]:
# ── Back-propagation parameters ──────────────────────────────────────────────
# Sources placed at Tx-Rx midpoints (x_traces) — consistent with exploding-
# reflector half-velocity (v_mig = v/2) used in gazdag_migration.
slugs_all = ['baseline', '2lambda', '1lambda', '0p5lambda', '0p25lambda', '0p125lambda', '0p0625lambda', '0p03125lambda']
STRIDE   = 1    # use every Nth trace — reduces gprMax source count
N_SNAP   = 30   # number of snapshots in the focus window
SNAP_WIN = 1.0  # [ns] window before focus time to capture

# ── Generate files for all 8 datasets ────────────────────────────────────────
print(f'Back-propagation file generation  (stride={STRIDE}, {N_SNAP} snapshots)\n')
bp_paths = {}
for label, slug, raw in zip(labels_all, slugs_all, outputs_static):
    tapered, _ = preprocess(raw, dt_ns, t0_ns, time_ns)
    in_path, n_src, n_snaps, t_focus = write_backprop_files(
        STUDY_ROOT, label, slug, tapered, dt_ns, x_traces,
        t0_ns, eps_r, v_ice, stride=STRIDE, n_snap=N_SNAP, snap_win=SNAP_WIN
    )
    bp_paths[label] = in_path
    exc_mb = (in_path.parent / 'excitation.txt').stat().st_size / 1e6
    print(f'  [{label}]  {n_src} sources  |  {n_snaps} snapshots  |  '
          f'focus={t_focus:.2f} ns  |  excitation={exc_mb:.1f} MB')
    print(f'           {in_path}')

In [ ]:
import pyvista

# ── Load focus frame for all 7 datasets ────────────────────────────────────────────
T_ns_bp    = n_t * dt_ns
t_focus_ns = T_ns_bp - t0_ns
t_start_ns = max(0.0, t_focus_ns - SNAP_WIN)
dt_s       = dt_ns * 1e-9
snap_step_ref = max(1, int((T_ns_bp*1e-9 - t_start_ns*1e-9) / (max(1, N_SNAP - 1) * dt_s)))

focus_frames = {}
for i, (label, slug) in enumerate(zip(labels_all, slugs_all)):
    snap_dir   = STUDY_ROOT / 'backprop' / slug / f'backprop_{slug}_snaps'
    snap_files = sorted(
        snap_dir.glob('bp_snap*.vti'),
        key=lambda p: int(p.stem.replace('bp_snap', ''))
    )
    if not snap_files:
        print(f'[{label}] No snapshots in {snap_dir.name} — skipping')
        continue

    snap_times_ns = t_start_ns + np.arange(len(snap_files)) * snap_step_ref * dt_ns
    idx_focus     = min(int(np.argmin(np.abs(snap_times_ns - t_focus_ns))) + 4,
                        len(snap_files) - 1)

    snaps_mag, snaps_ez = [], []
    for p in snap_files:
        mesh   = pyvista.read(str(p))
        e_data = np.array(mesh['E-field'])
        snaps_mag.append(np.linalg.norm(e_data, axis=1).reshape(1000, 4000))
        snaps_ez.append(e_data[:, 2].reshape(1000, 4000))

    focus_frames[label] = {
        'mag':      np.stack(snaps_mag)[idx_focus],
        'ez':       np.stack(snaps_ez)[idx_focus],
        't_actual': snap_times_ns[idx_focus],
        'i':        i,
    }
    print(f'[{label}]  {len(snap_files)} snaps  |  focus idx={idx_focus}'
          f'  t={focus_frames[label]["t_actual"]:.3f} ns')

extent_full = [0, 4.0, 0, 1]
margin_x    = 0.4
margin_y    = 0.12

# Zoom x-window fixed to cover the widest (2λ) shift in all shared-axis panels
x_zoom_lo = x_scatterer_1[0] - margin_x
x_zoom_hi = x_scatterer_1[1] + margin_x   # x_scatterer_1[1] = 2λ position

# Current-position marker looks up each frame's own stored scenario index
# (frame['i']), not the panel's plotting-loop position, so it stays correct
# even if a dataset without snapshots gets skipped above.
_marker_x_bp = lambda i, label: x_scatterer_1[focus_frames[label]['i']]

# ── |E| magnitude — full extent + zoomed 2×4 grids ─────────────────────────────────
plot_wavefield_grid(
    {label: frame['mag'] for label, frame in focus_frames.items()}, extent_full,
    field='magnitude', ncols=4, y_surface=y_surface,
    title=f'Back-Propagation |E| — All 7 Datasets  |  focus at {t_focus_ns:.2f} ns',
    marker_x=_marker_x_bp, marker_y=y_scatterer, marker_x_baseline=x_scatterer_1[0],
    zooms=[
        (None, None, ''),
        ((x_zoom_lo, x_zoom_hi), (y_scatterer - margin_y, y_scatterer + margin_y), '(zoomed)'),
    ],
)
plt.show()

# ── Ez component — full extent + zoomed 2×4 grids ───────────────────────────────────
plot_wavefield_grid(
    {label: frame['ez'] for label, frame in focus_frames.items()}, extent_full,
    field='signed', ncols=4, y_surface=y_surface,
    title=f'Back-Propagation Ez — All 7 Datasets  |  focus at {t_focus_ns:.2f} ns',
    marker_x=_marker_x_bp, marker_y=y_scatterer, marker_x_baseline=x_scatterer_1[0],
    zooms=[
        (None, None, ''),
        ((x_zoom_lo, x_zoom_hi), (y_scatterer - margin_y, y_scatterer + margin_y), '(zoomed)'),
    ],
)
plt.show()

In [ ]:
labels = ['2λ', '1λ', '½λ', '¼λ', '⅛λ', '¹⁄₁₆λ', '¹⁄₃₂λ']

In [ ]:
# ── Back-propagation timelapse differences (Ez − Ez_baseline) ─────────────────────
bp_ez_diff = {label: focus_frames[label]['ez'] - focus_frames['Baseline']['ez']
              for label in labels}

plot_wavefield_grid(
    bp_ez_diff, extent_full, field='signed', ncols=4,
    title=f'Back-Propagation — TimeLapse Differences Ez (Ez − Ez_baseline)  |  focus at {t_focus_ns:.2f} ns',
    marker_x=lambda i, label: x_scatterer_1[i + 1],
    marker_y=y_scatterer,
    marker_x_baseline=x_scatterer_1[0],
    zooms=[
        (None, None, ''),
        ((x_zoom_lo, x_zoom_hi), (y_scatterer - margin_y, y_scatterer + margin_y), '(zoomed)'),
    ],
)
plt.show()

# Analysis

In [ ]:
from scipy.interpolate import RegularGridInterpolator

# ── Collect timelapse difference images: rows = separations, cols = methods ─
methods = ['Kirchhoff', 'Gazdag', 'LSM (CGLS)', 'Back-prop']
n_m, n_s = len(methods), len(labels)
imgs = [[None] * n_m for _ in range(n_s)]

for i, lbl in enumerate(labels):
    # Kirchhoff: use precomputed diff dict; fall back to inline if stale kernel
    try:
        imgs[i][0] = migrated_diff[lbl]
    except (KeyError, NameError):
        if 'Baseline' in migrated and lbl in migrated:
            imgs[i][0] = migrated[lbl] - migrated['Baseline']
    # Gazdag: same pattern
    try:
        imgs[i][1] = migrated_gz_diff[lbl]
    except (KeyError, NameError):
        if 'Baseline' in migrated_gz and lbl in migrated_gz:
            imgs[i][1] = migrated_gz[lbl] - migrated_gz['Baseline']

# LSM (CGLS) — 2λ only (if available). m_cgls/m_adj are never actually
# defined in this notebook, so this always falls through to `pass` and the
# column renders as an N/A placeholder -- kept only for index compatibility
# with the save cell below (see note above).
try:
    imgs[0][2] = m_cgls
except NameError:
    try:
        imgs[0][2] = m_adj
    except NameError:
        pass

# Back-propagation — reproject bp_ez_diff (already in memory) to migration grid
x_ax_bp = np.linspace(0, 4.0, 4000)
y_ax_bp = np.linspace(0, 1.0, 1000)
y_mig   = y_surface - z_img
Yq, Xq  = np.meshgrid(y_mig, x_traces, indexing='ij')

for i, lbl in enumerate(labels):
    if lbl not in bp_ez_diff:
        print(f'[{lbl}] bp_ez_diff missing — skipping')
        continue
    interp = RegularGridInterpolator(
        (y_ax_bp, x_ax_bp), bp_ez_diff[lbl],
        method='linear', bounds_error=False, fill_value=0.0
    )
    imgs[i][3] = interp((Yq, Xq))
    print(f'[{lbl}] back-prop diff reprojected  shape={imgs[i][3].shape}')

# ── Scatterer depth coordinates ──────────────────────────────────────────────
# Kirchhoff/Gazdag: z_top = depth to cylinder top [m]
# Back-prop: gprMax y-axis runs 0–1 m from bottom; convert to depth from surface
bp_z_top = 0.9 - y_scatterer   # = z_scatterer ≈ 0.676 m

# ── Comparison grid: signed amplitude ────────────────────────────────────────
extent_mig = [x_traces[0], x_traces[-1], z_img[-1], z_img[0]]
xlim = (1.3, 2.7)
ylim = (0.80, 0.54)

def _marker_z_cmp(i, j):
    return bp_z_top if j == 3 else z_top

plot_method_comparison_grid(
    imgs, extent_mig, methods, labels,
    title=f'TimeLapse Migration Comparison — Signed Amplitude  |  f_c={f_c_GHz} GHz  |  aperture={ANGLE_AP}°',
    envelope=False,
    marker_x=lambda i, j: x_scatterer_1[i + 1],
    marker_z=_marker_z_cmp,
    xlim=xlim, ylim=ylim,
)
plt.show()

In [ ]:
# ── Normalised lateral PSF of timelapse difference at scatterer depth ────────
# Rows = scenario (2λ … ¹⁄₁₆λ), Columns = method
# Each panel: horizontal slice of the timelapse difference image at z = marker_z,
# normalised to its absolute peak, showing how well each method localises the change.
# Blue  = signed amplitude (normalised to peak)
# Red   = Hilbert envelope
# Green dashed = timelapsed scatterer x-position
# Green dotted  = baseline scatterer x-position

x_win = 3.0 * wavelength   # half-window around scatterer x-position [m]

def _psf_profile(i, j):
    img = imgs[i][j]
    if img is None:
        return None
    marker_z = bp_z_top if j == 3 else z_top
    iz = int(np.argmin(np.abs(z_img - marker_z)))
    profile = img[iz, :].copy()
    peak = np.max(np.abs(profile))
    if peak > 0:
        profile /= peak
    return profile

profiles = [[_psf_profile(i, j) for j in range(n_m)] for i in range(n_s)]
# Same [timelapsed, baseline] reference pair drawn in every method column of a row.
marker_positions = [[[x_scatterer_1[i + 1], x_scatterer_1[0]] for _ in range(n_m)]
                     for i in range(n_s)]

fig_psf, axes_psf = plot_psf_grid(
    profiles, x_traces, methods, labels, orientation='lateral',
    marker_positions=marker_positions, ylim=(-1.15, 1.4),
    title=('Normalised Lateral PSF — TimeLapse Difference at True Scatterer Depth\n'
           'blue = amplitude  |  red = Hilbert envelope  '
           '|  green dashed = x_s1 timelapsed  |  green dotted = x_s1 baseline'),
)
# Per-row x-window centred on that row's timelapsed scatterer position -- not
# expressible via plot_psf_grid's single shared xlim=, so applied here on the
# returned axes before plt.show() (sharex=False for orientation='lateral').
for i in range(n_s):
    for j in range(n_m):
        axes_psf[i, j].set_xlim(x_scatterer_1[i + 1] - x_win, x_scatterer_1[i + 1] + x_win)
axes_psf[0, 0].legend(fontsize=7, loc='upper right')
plt.show()

# Save Migrated Results as np arrays dictionary

In [ ]:
# ── Build structured numpy representation of all migrated difference results ──────
# imgs[scenario_idx][method_idx] is populated by the analysis cell above.
#   scenarios = labels   →  ["2λ", "1λ", "½λ", "¼λ", "⅛λ", "¹⁄₁₆λ"]
#   methods              →  ["Kirchhoff", "Gazdag", "LSM (CGLS)", "Back-prop"]
#   each image: shape (n_z, n_x) — timelapse difference (monitor − baseline)
#   LSM (CGLS) was run for the 2λ scenario only — None elsewhere.

_ref_shape = imgs[0][0].shape   # (n_z, n_x) reference from Kirchhoff 2λ

def _safe_stack(method_idx):
    planes = []
    for i in range(n_s):
        arr = imgs[i][method_idx]
        planes.append(arr.astype(float) if arr is not None
                      else np.full(_ref_shape, np.nan))
    return np.stack(planes, axis=0)

diff_kirchhoff = _safe_stack(0)   # (6, n_z, n_x)  timelapse differences
diff_gazdag    = _safe_stack(1)
diff_backprop  = _safe_stack(3)
lsm_cgls_2lam  = imgs[0][2].astype(float) if imgs[0][2] is not None else None

# ── Save to disk ──────────────────────────────────────────────────────────────
save_path = STUDY_ROOT / "difference_migrated_results.npz"

payload = dict(
    kirchhoff_diff    = diff_kirchhoff,                                 # (6, n_z, n_x)
    gazdag_diff       = diff_gazdag,
    backprop_diff     = diff_backprop,
    scenarios         = np.array(labels,  dtype="U20"),                 # (6,)  string
    methods           = np.array(methods, dtype="U20"),                 # (4,)  string
    separation_lambda = np.array([2, 1, 0.5, 0.25, 0.125, 0.0625]),    # (6,)  float
    x_traces          = x_traces,                                       # (n_x,) m
    z_img             = z_img,                                          # (n_z,) m
    x_s1              = np.array(x_scatterer_1),                        # (7,) m
    x_s2              = np.float64(x_scatterer_2),
    z_scatterer       = np.float64(z_scatterer),
    z_top             = np.float64(z_top),
)
if lsm_cgls_2lam is not None:
    payload["lsm_cgls_2lambda_diff"] = lsm_cgls_2lam   # (n_z, n_x) — 2λ only

np.savez_compressed(save_path, **payload)

# ── In-memory convenience dict ────────────────────────────────────────────────
mig_diff = {
    "Kirchhoff": {lbl: diff_kirchhoff[i] for i, lbl in enumerate(labels)},
    "Gazdag":    {lbl: diff_gazdag[i]    for i, lbl in enumerate(labels)},
    "Back-prop": {lbl: diff_backprop[i]  for i, lbl in enumerate(labels)},
}
if lsm_cgls_2lam is not None:
    mig_diff["LSM (CGLS)"] = {labels[0]: lsm_cgls_2lam}

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"Saved → {save_path}")
print(f"\nDifference arrays  (n_scenarios={n_s}, n_z={_ref_shape[0]}, n_x={_ref_shape[1]}):")
for name, arr in [("kirchhoff_diff", diff_kirchhoff),
                  ("gazdag_diff",    diff_gazdag),
                  ("backprop_diff",  diff_backprop)]:
    absent = int(np.isnan(arr).all(axis=(1, 2)).sum())
    note   = f"  ({absent} scenario(s) absent → NaN)" if absent else ""
    print(f"  {name:<18}  {arr.shape}{note}")
if lsm_cgls_2lam is not None:
    print(f"  lsm_cgls_diff     {lsm_cgls_2lam.shape}  (2λ only)")

print(f"\nAvailability  ✓ = computed   — = not run")
col_w = 14
print("  " + f"{'':>8}  " + "".join(f"{m:>{col_w}}" for m in methods))
for i, lbl in enumerate(labels):
    row = "  " + f"{lbl:>8}  "
    row += "".join(f"{'✓':>{col_w}}" if imgs[i][j] is not None else f"{'—':>{col_w}}"
                   for j in range(n_m))
    print(row)


In [ ]:
# ── Save normal (non-difference) migrated results from all methods ────────────
# Stacks all 7 scenarios (Baseline + 6 shifts) from migrated, migrated_gz,
# and focus_frames (back-prop), saving to migrated_results.npz.
# Index 0 = Baseline, indices 1-6 = shift scenarios matching labels_all[1:].

# ── Kirchhoff & Gazdag: direct stack over labels_all ─────────────────────────
kirchhoff_all = np.stack([migrated[lbl].astype(float)    for lbl in labels_all], axis=0)  # (7, n_z, n_x)
gazdag_all    = np.stack([migrated_gz[lbl].astype(float) for lbl in labels_all], axis=0)

# ── Back-prop: reproject focus_frames ez onto migration grid ──────────────────
x_ax_bp = np.linspace(0, 4.0, 4000)
y_ax_bp = np.linspace(0, 1.0, 1000)
y_mig   = y_surface - z_img
Yq, Xq  = np.meshgrid(y_mig, x_traces, indexing='ij')

from scipy.interpolate import RegularGridInterpolator as _RGI
backprop_planes = []
for lbl in labels_all:
    if lbl in focus_frames:
        interp = _RGI((y_ax_bp, x_ax_bp), focus_frames[lbl]['ez'],
                      method='linear', bounds_error=False, fill_value=0.0)
        backprop_planes.append(interp((Yq, Xq)).astype(float))
    else:
        backprop_planes.append(np.full((len(z_img), len(x_traces)), np.nan))
        print(f'[{lbl}] focus_frames missing — filled with NaN')
backprop_all = np.stack(backprop_planes, axis=0)  # (7, n_z, n_x)

# ── Save to disk ──────────────────────────────────────────────────────────────
save_path_mig = STUDY_ROOT / "migrated_results.npz"

np.savez_compressed(
    save_path_mig,
    kirchhoff         = kirchhoff_all,                                   # (9, n_z, n_x)
    gazdag            = gazdag_all,
    backprop          = backprop_all,
    scenarios         = np.array(labels_all, dtype="U20"),               # (9,)  Baseline + 8 shifts
    separation_lambda = np.array([0, 2, 1, 0.5, 0.25, 0.125, 0.0625, 0.03125]),  # (9,)  0 = baseline
    x_traces          = x_traces,
    z_img             = z_img,
    x_s1              = np.array(x_scatterer_1),
    x_s2              = np.float64(x_scatterer_2),
    z_scatterer       = np.float64(z_scatterer),
    z_top             = np.float64(z_top),
)

print(f"Saved → {save_path_mig}")
print(f"\nStacked arrays  (n_scenarios=9, n_z={kirchhoff_all.shape[1]}, n_x={kirchhoff_all.shape[2]}):")
for name, arr in [("kirchhoff",  kirchhoff_all),
                  ("gazdag",     gazdag_all),
                  ("backprop",   backprop_all)]:
    absent = int(np.isnan(arr).all(axis=(1, 2)).sum())
    note   = f"  ({absent} scenario(s) absent → NaN)" if absent else ""
    print(f"  {name:<12}  {arr.shape}{note}")
print(f"\nUsage examples:")
print(f"  d = np.load(str(STUDY_ROOT / 'migrated_results.npz'), allow_pickle=False)")
print(f"  d['kirchhoff'][0]   # Kirchhoff Baseline → shape (n_z, n_x)")
print(f"  d['gazdag'][3]      # Gazdag ¼λ          → shape (n_z, n_x)")


# Application to Noisy Data

## Save Unmigrated Noisy B-scans

In [ ]:
# -- Save non-migrated (noisy, background-subtracted) B-scan data --------------
# Mirrors the "Saving the Data" step above, but for the Laplace-noise-augmented
# B-scans (data_noisy) created in the "Create Noisy Data" section.

noisy_static_all = np.stack([d.astype(float) for _, d in data_noisy], axis=0)  # (8, n_t, n_traces)

save_path_noisy_static = STUDY_ROOT / "static_results_noisy.npz"

np.savez_compressed(
    save_path_noisy_static,
    data_static       = noisy_static_all,                                    # (8, n_t, n_traces)
    scenarios         = np.array(labels_all, dtype="U20"),                   # (8,)  Baseline + 7 shifts
    separation_lambda = np.array([0, 2, 1, 0.5, 0.25, 0.125, 0.0625, 0.03125]),  # (8,)  0 = baseline
    x_traces          = x_traces,
    time_ns           = time_ns,
    dt                = np.float64(dt),
    x_s1              = np.array(x_scatterer_1),
    x_s2              = np.float64(x_scatterer_2),
    z_scatterer       = np.float64(z_scatterer),
    z_top             = np.float64(z_top),
    noise_level       = np.float64(NOISE_LEVEL),
)

print(f"Saved -> {save_path_noisy_static}")
print(f"\nStacked array  (n_scenarios={noisy_static_all.shape[0]}, n_t={noisy_static_all.shape[1]}, n_tr={noisy_static_all.shape[2]}):")
print(f"  data_static   {noisy_static_all.shape}")


## Kirchhoff Migration (Noisy)

In [ ]:
# -- Kirchhoff migration on all 8 noisy datasets --------------------------------
migrated_noisy = {}

for label, raw in zip(labels_all, outputs_noisy):
    _, shifted = preprocess(raw, dt_ns, t0_ns, time_ns)   # (n_tr, n_t), t0-shifted
    print(f'[{label}] Running Kirchhoff (noisy) ...', end=' ', flush=True)
    t0 = _time.perf_counter()
    img = PylopsKirchoffMigration(
        shifted, time_ns, x_traces, v_ice, z_img,
        f0=f_c_GHz, angleaperture=ANGLE_AP
    )
    print(f'{_time.perf_counter()-t0:.1f} s  shape={img.shape}')
    migrated_noisy[label] = img


In [ ]:
extent_mig = [x_traces[0], x_traces[-1], z_img[-1], z_img[0]]

# -- Plot: full extent + zoomed ---------------------------------------------------
plot_migrated_grid(
    migrated_noisy, extent_mig, ncols=4,
    title=f'Kirchhoff Migration — All 8 Noisy Datasets  |  f_c={f_c_GHz} GHz  |  aperture={ANGLE_AP} deg  |  z_top={z_top:.3f} m',
    marker_x=lambda i, label: x_scatterer_1[i],
    marker_z=z_top,
    marker_x_baseline=x_scatterer_1[0],
    zooms=[
        (None, None, ''),
        ((1.5, 2.5), (0.75, 0.55), '(noisy, zoomed)'),
    ],
)
plt.show()

In [ ]:
# -- Kirchhoff timelapse differences (noisy migrated - noisy migrated_baseline) --
migrated_diff_noisy = {label: migrated_noisy[label] - migrated_noisy['Baseline'] for label in labels}

plot_migrated_grid(
    migrated_diff_noisy, extent_mig, ncols=4,
    title=f'Kirchhoff Migration — Noisy TimeLapse Differences (migrated − migrated_baseline)  |  f_c={f_c_GHz} GHz',
    marker_x=lambda i, label: x_scatterer_1[i + 1],
    marker_z=z_top,
    marker_x_baseline=x_scatterer_1[0],
    zooms=[
        (None, None, ''),
        ((1.5, 2.5), (0.75, 0.55), '(zoomed)'),
    ],
)
plt.show()

## Gazdag Phase-Shift Migration (Noisy)

In [ ]:
# -- Gazdag phase-shift migration on all 8 noisy datasets -----------------------
migrated_gz_noisy = {}

for label, raw in zip(labels_all, outputs_noisy):
    _, shifted = preprocess(raw, dt_ns, t0_ns, time_ns)   # (n_tr, n_t), t0-shifted
    print(f'[{label}] Running Gazdag (noisy) ...', flush=True) 
    t0 = _time.perf_counter()
    img = gazdag_migration(
        shifted.T,       # (n_t, n_tr) - time first
        x_traces, time_ns, z_img, v_ice
    )
    print(f'  Done in {_time.perf_counter()-t0:.1f} s  shape={img.shape}')
    migrated_gz_noisy[label] = img


In [ ]:
# -- Plot: full extent -------------------------------------------------------------
plot_migrated_grid(
    migrated_gz_noisy, extent_mig, ncols=4,
    title=f'Gazdag Phase-Shift Migration — All 8 Noisy Datasets  |  f_c={f_c_GHz} GHz  |  z_top={z_top:.3f} m',
    marker_x=lambda i, label: x_scatterer_1[i],
    marker_z=z_top,
    marker_x_baseline=x_scatterer_1[0],
)
plt.show()

# -- Plot: zoomed on scatterer region (tighter vmax/20 color scale) ----------------
plot_migrated_grid(
    migrated_gz_noisy, extent_mig, ncols=4,
    title=f'Gazdag Phase-Shift Migration — Noisy (zoomed)  |  f_c={f_c_GHz} GHz',
    marker_x=lambda i, label: x_scatterer_1[i],
    marker_z=z_top,
    marker_x_baseline=x_scatterer_1[0],
    vmax_headroom=0.04,  # reproduces the original vmax/20 zoom scaling (0.8/20)
    zooms=[((1.5, 2.5), (0.75, 0.55), '')],
)
plt.show()

In [ ]:
# -- Gazdag timelapse differences (noisy migrated - noisy migrated_baseline) ------
migrated_gz_diff_noisy = {label: migrated_gz_noisy[label] - migrated_gz_noisy['Baseline'] for label in labels}

plot_migrated_grid(
    migrated_gz_diff_noisy, extent_mig, ncols=3,
    title=f'Gazdag Migration — Noisy TimeLapse Differences (migrated − migrated_baseline)  |  f_c={f_c_GHz} GHz',
    marker_x=lambda i, label: x_scatterer_1[i + 1],
    marker_z=z_top,
    marker_x_baseline=x_scatterer_1[0],
)
plt.show()

# -- Zoomed (tighter vmax/20 color scale, matching the full-image cell above) -----
plot_migrated_grid(
    migrated_gz_diff_noisy, extent_mig, ncols=3,
    title='Gazdag Migration — Noisy TimeLapse Differences (zoomed)',
    marker_x=lambda i, label: x_scatterer_1[i + 1],
    marker_z=z_top,
    marker_x_baseline=x_scatterer_1[0],
    vmax_headroom=0.04,
    zooms=[((1.5, 2.5), (0.75, 0.55), '')],
)
plt.show()

## Back-Propagation Migration (Noisy)

Same time-reversal approach as the clean-data section above, but the noisy
background-subtracted traces (`outputs_noisy`) are re-injected as sources instead
of the clean ones. Kirchhoff/Gazdag on noisy data is a pure post-processing step
on already-loaded arrays, but back-propagation requires a brand new gprMax
**forward** (FDTD) simulation per scenario, since the excitation file *is* the
time-reversed noisy trace data.

**Sign-bit time reversal:** the first pass (peak-normalised excitation, same as
the clean-data section) showed the Laplace noise spikes acting as their own
competing point sources during back-propagation, interfering at the true source
locations instead of being suppressed by destructive interference. Spatial
focusing during back-propagation is governed almost entirely by *phase*
(zero-crossings), not amplitude — so instead of injecting the peak-normalised
time-reversed wavefield $u(x,\tau)$, we inject only its sign:

$$u_{\text{sign}}(x, \tau) = \operatorname{sign}\big(u(x, \tau)\big)$$

This keeps every zero-crossing and phase trend of the GPR wavelet intact while
squashing the Laplace spikes down to the same $\pm 1$ as the coherent signal —
stripping them of the outsized amplitude that let them dominate the
back-propagated wavefield. `write_backprop_files(..., sign_bit=True)` below
implements this (see `helper_functions/migration.py`); the clean-data section
above is left on the default peak-normalised mode since it has no noise to
suppress.

Files are written to `timelapse_study/backprop/<slug>_noisy/` — slug suffixed with
`_noisy` to keep them alongside, but distinct from, the clean-data runs already in
`backprop/`.

**This notebook only generates the `.in` files below** — run each through gprMax
externally to produce the `.vti` snapshots the load/plot cells expect (the
clean-data section above has the same split: no cell in this notebook calls
`api()` for back-propagation, snapshots are assumed to already exist on disk). E.g.:

```
python -m gprMax timelapse_study/backprop/2lambda_noisy/backprop_2lambda_noisy.in
```

Run all 8 `.in` files (`baseline_noisy`, `2lambda_noisy`, ..., `0p03125lambda_noisy`),
then re-run the two cells below.

In [ ]:
# -- Back-propagation file generation (noisy) ------------------------------------
# Mirrors the clean-data cell above, but re-injects the noisy background-
# subtracted traces (outputs_noisy) using sign-bit time reversal (sign_bit=True --
# see markdown above) and writes to '<slug>_noisy' folders so the clean-data
# .in/.out/snapshot files are never touched.
slugs_noisy = [f'{slug}_noisy' for slug in slugs_all]

# Sign-bit traces still carry real signal bandwidth that can exceed gprMax's
# numerical-dispersion limit (cells/wavelength) for the finer lambda-fraction
# separations, causing a 'Non-physical wave propagation' error. Low-pass filter
# each excitation file right after writing it so every .in file is runnable.
# eps_r*4 / dx=0.001 mirror the half-velocity material and grid hardcoded in
# write_backprop_files() -- see helper_functions/migration.py.
cutoff_hz = dispersion_limited_cutoff(eps_r=4.0 * eps_r, dx=0.001)

print(f'Back-propagation file generation (noisy, sign-bit)  (stride={STRIDE}, {N_SNAP} snapshots)' + chr(10))
bp_paths_noisy = {}
for label, slug, raw in zip(labels_all, slugs_noisy, outputs_noisy):
    tapered, _ = preprocess(raw, dt_ns, t0_ns, time_ns)
    in_path, n_src, n_snaps, t_focus = write_backprop_files(
        STUDY_ROOT, f'{label} (noisy, sign-bit)', slug, tapered, dt_ns, x_traces,
        t0_ns, eps_r, v_ice, stride=STRIDE, n_snap=N_SNAP, snap_win=SNAP_WIN,
        sign_bit=True
    )
    lowpass_filter_excitation(in_path.parent / 'excitation.txt', cutoff_hz)
    bp_paths_noisy[label] = in_path
    exc_mb = (in_path.parent / 'excitation.txt').stat().st_size / 1e6
    print(f'  [{label}]  {n_src} sources  |  {n_snaps} snapshots  |  '
          f'focus={t_focus:.2f} ns  |  excitation={exc_mb:.1f} MB  |  '
          f'low-pass={cutoff_hz/1e9:.1f} GHz')
    print(f'           {in_path}')

print(chr(10) + 'NOTE: .in files only -- run each through gprMax externally to produce the '
      '.vti snapshots the cells below expect, then re-run them.')

In [ ]:
# -- Sign-bit time-reversed B-scans + their frequency spectra (noisy) -----------
# Recreates the same time-reversal + sign-bit transform used inside
# write_backprop_files(..., sign_bit=True) above, purely for visualisation:
# row 1 shows the resulting B-scan that gets injected into gprMax as the
# source excitation; row 2 shows its frequency spectrum (mean over traces),
# illustrating the broadband content that lowpass_filter_excitation() trims.
sign_bit_bscans = {}
for label, raw in zip(labels_all, outputs_noisy):
    tapered, _ = preprocess(raw, dt_ns, t0_ns, time_ns)   # (n_tr, n_t)
    data_rev = tapered[::STRIDE, ::-1]
    sign_bit_bscans[label] = np.sign(data_rev)            # (n_src, n_t)

freqs_ghz = np.fft.rfftfreq(n_t, d=dt_ns)

fig, axes = plt.subplots(2, len(sign_bit_bscans), figsize=(25, 8))

plot_bscan_grid(
    [(f'{label} (sign-bit)', sb.T) for label, sb in sign_bit_bscans.items()],
    x_traces, time_ns, axes=axes[0], vmax=1.0, shared_colorbar=False,
)

# norm='forward' standardizes this against every other spectrum cell in the
# notebook (the original version of this cell used axis=1/no norm, which
# plot_spectrum_grid's expected np.fft.rfft(..., axis=0, norm='forward')
# convention fixes here as a side effect of the refactor).
spectra_sign_bit = [[
    (f'{label} (sign-bit)', np.abs(np.fft.rfft(sb.T, axis=0, norm='forward')).mean(axis=1))
    for label, sb in sign_bit_bscans.items()
]]
plot_spectrum_grid(spectra_sign_bit, freqs_ghz, title=None, axes=axes[1])

fig.suptitle('Sign-Bit Time-Reversed Excitation — B-scans and Spectra (Noisy)',
             fontsize=12, fontweight='bold', y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
import pyvista

In [ ]:
# -- Load focus frame for all 8 noisy datasets (if snapshots exist) --------------
T_ns_bp    = n_t * dt_ns
t_focus_ns = T_ns_bp - t0_ns
t_start_ns = max(0.0, t_focus_ns - SNAP_WIN)
dt_s       = dt_ns * 1e-9
snap_step_ref = max(1, int((T_ns_bp*1e-9 - t_start_ns*1e-9) / (max(1, N_SNAP - 1) * dt_s)))

focus_frames_noisy = {}
for i, (label, slug) in enumerate(zip(labels_all, slugs_noisy)):
    snap_dir   = STUDY_ROOT / 'backprop' / slug / f'backprop_{slug}_snaps'
    snap_files = sorted(
        snap_dir.glob('bp_snap*.vti'),
        key=lambda p: int(p.stem.replace('bp_snap', ''))
    )
    if not snap_files:
        print(f'[{label}] No snapshots in {snap_dir.name} -- run the .in file through gprMax first, skipping')
        continue

    snap_times_ns = t_start_ns + np.arange(len(snap_files)) * snap_step_ref * dt_ns
    idx_focus     = min(int(np.argmin(np.abs(snap_times_ns - t_focus_ns))) + 4,
                        len(snap_files) - 1)

    snaps_mag, snaps_ez = [], []
    for p in snap_files:
        mesh   = pyvista.read(str(p))
        e_data = np.array(mesh['E-field'])
        snaps_mag.append(np.linalg.norm(e_data, axis=1).reshape(1000, 4000))
        snaps_ez.append(e_data[:, 2].reshape(1000, 4000))

    focus_frames_noisy[label] = {
        'mag':      np.stack(snaps_mag)[idx_focus],
        'ez':       np.stack(snaps_ez)[idx_focus],
        't_actual': snap_times_ns[idx_focus],
        'i':        i,
    }
    print(f'[{label}]  {len(snap_files)} snaps  |  focus idx={idx_focus}'
          f'  t={focus_frames_noisy[label]["t_actual"]:.3f} ns')

if not focus_frames_noisy:
    print('\nNo noisy back-propagation snapshots found yet -- run the .in files '
          'generated above through gprMax, then re-run this cell.')
else:
    extent_full = [0, 4.0, 0, 1]
    margin_x    = 0.4
    margin_y    = 0.12
    x_zoom_lo = x_scatterer_1[0] - margin_x
    x_zoom_hi = x_scatterer_1[1] + margin_x

    _marker_x_bp_noisy = lambda i, label: x_scatterer_1[focus_frames_noisy[label]['i']]

    # |E| magnitude -- full extent + zoomed 2x4 grids
    plot_wavefield_grid(
        {label: frame['mag'] for label, frame in focus_frames_noisy.items()}, extent_full,
        field='magnitude', ncols=4, y_surface=y_surface,
        title=f'Back-Propagation |E| (Noisy) — All Datasets  |  focus at {t_focus_ns:.2f} ns',
        marker_x=_marker_x_bp_noisy, marker_y=y_scatterer, marker_x_baseline=x_scatterer_1[0],
        zooms=[
            (None, None, ''),
            ((x_zoom_lo, x_zoom_hi), (y_scatterer - margin_y, y_scatterer + margin_y), '(noisy, zoomed)'),
        ],
    )
    plt.show()

    # Ez component -- full extent + zoomed 2x4 grids
    plot_wavefield_grid(
        {label: frame['ez'] for label, frame in focus_frames_noisy.items()}, extent_full,
        field='signed', ncols=4, y_surface=y_surface,
        title=f'Back-Propagation Ez (Noisy) — All Datasets  |  focus at {t_focus_ns:.2f} ns',
        marker_x=_marker_x_bp_noisy, marker_y=y_scatterer, marker_x_baseline=x_scatterer_1[0],
        zooms=[
            (None, None, ''),
            ((x_zoom_lo, x_zoom_hi), (y_scatterer - margin_y, y_scatterer + margin_y), '(noisy, zoomed)'),
        ],
    )
    plt.show()

In [ ]:
# -- Back-propagation timelapse differences (noisy) (Ez_noisy - Ez_noisy_baseline) --
if not focus_frames_noisy or 'Baseline' not in focus_frames_noisy:
    print('Skipping noisy back-propagation timelapse-difference plot -- '
          'no noisy snapshots (or no Baseline snapshot) yet.')
else:
    bp_ez_diff_noisy = {label: focus_frames_noisy[label]['ez'] - focus_frames_noisy['Baseline']['ez']
                         for label in labels if label in focus_frames_noisy}

    plot_wavefield_grid(
        bp_ez_diff_noisy, extent_full, field='signed', ncols=4,
        title=f'Back-Propagation (Noisy) — TimeLapse Differences Ez  |  focus at {t_focus_ns:.2f} ns',
        marker_x=lambda i, label: x_scatterer_1[i + 1],
        marker_y=y_scatterer,
        marker_x_baseline=x_scatterer_1[0],
        zooms=[
            (None, None, ''),
            ((x_zoom_lo, x_zoom_hi), (y_scatterer - margin_y, y_scatterer + margin_y), '(zoomed)'),
        ],
    )
    plt.show()

## Save Migrated Results (Noisy, non-difference)

In [ ]:
# -- Save normal (non-difference) migrated results from the noisy datasets -----
# Stacks all 8 scenarios (Baseline + 7 shifts) from migrated_noisy, migrated_gz_noisy,
# and focus_frames_noisy (back-prop) into migrated_results_noisy.npz.
# Back-propagation is only included for scenarios whose .in file (generated above)
# has actually been run through gprMax externally and produced snapshots that the
# noisy-data loading cell above found (focus_frames_noisy) -- otherwise that
# scenario's plane is filled with NaN, same convention as migrated_results.npz.

kirchhoff_noisy_all = np.stack([migrated_noisy[lbl].astype(float)    for lbl in labels_all], axis=0)  # (8, n_z, n_x)
gazdag_noisy_all    = np.stack([migrated_gz_noisy[lbl].astype(float) for lbl in labels_all], axis=0)

# -- Back-prop (noisy): reproject focus_frames_noisy ez onto migration grid -----
x_ax_bp = np.linspace(0, 4.0, 4000)
y_ax_bp = np.linspace(0, 1.0, 1000)
y_mig   = y_surface - z_img
Yq, Xq  = np.meshgrid(y_mig, x_traces, indexing='ij')

from scipy.interpolate import RegularGridInterpolator as _RGI
backprop_noisy_planes = []
for lbl in labels_all:
    if lbl in focus_frames_noisy:
        interp = _RGI((y_ax_bp, x_ax_bp), focus_frames_noisy[lbl]['ez'],
                      method='linear', bounds_error=False, fill_value=0.0)
        backprop_noisy_planes.append(interp((Yq, Xq)).astype(float))
    else:
        backprop_noisy_planes.append(np.full((len(z_img), len(x_traces)), np.nan))
backprop_noisy_all = np.stack(backprop_noisy_planes, axis=0)  # (8, n_z, n_x)

save_path_mig_noisy = STUDY_ROOT / "migrated_results_noisy.npz"

np.savez_compressed(
    save_path_mig_noisy,
    kirchhoff         = kirchhoff_noisy_all,
    gazdag            = gazdag_noisy_all,
    backprop          = backprop_noisy_all,
    scenarios         = np.array(labels_all, dtype="U20"),               # (8,)  Baseline + 7 shifts
    separation_lambda = np.array([0, 2, 1, 0.5, 0.25, 0.125, 0.0625, 0.03125]),  # (8,)  0 = baseline
    x_traces          = x_traces,
    z_img             = z_img,
    x_s1              = np.array(x_scatterer_1),
    x_s2              = np.float64(x_scatterer_2),
    z_scatterer       = np.float64(z_scatterer),
    z_top             = np.float64(z_top),
    noise_level       = np.float64(NOISE_LEVEL),
)

print(f"Saved -> {save_path_mig_noisy}")
print(f"\nStacked arrays  (n_scenarios=8, n_z={kirchhoff_noisy_all.shape[1]}, n_x={kirchhoff_noisy_all.shape[2]}):")
for name, arr in [("kirchhoff", kirchhoff_noisy_all),
                  ("gazdag",    gazdag_noisy_all),
                  ("backprop",  backprop_noisy_all)]:
    absent = int(np.isnan(arr).all(axis=(1, 2)).sum())
    note   = f"  ({absent} scenario(s) absent -> NaN)" if absent else ""
    print(f"  {name:<12}  {arr.shape}{note}")
print(f"\nUsage examples:")
print(f"  d = np.load(str(STUDY_ROOT / 'migrated_results_noisy.npz'), allow_pickle=False)")
print(f"  d['kirchhoff'][0]   # Kirchhoff Baseline (noisy) -> shape (n_z, n_x)")
print(f"  d['gazdag'][3]      # Gazdag 1/4 lambda (noisy)  -> shape (n_z, n_x)")
print(f"  d['backprop'][1]    # Back-prop 2 lambda (noisy) -> shape (n_z, n_x), NaN if not yet run")

# Analysis (Noisy Data)

In [ ]:
from scipy.interpolate import RegularGridInterpolator

# ── Collect noisy timelapse difference images: rows = separations, cols = methods ─
imgs_noisy = [[None] * n_m for _ in range(n_s)]

for i, lbl in enumerate(labels):
    # Kirchhoff: use precomputed diff dict; fall back to inline if stale kernel
    try:
        imgs_noisy[i][0] = migrated_diff_noisy[lbl]
    except (KeyError, NameError):
        if 'Baseline' in migrated_noisy and lbl in migrated_noisy:
            imgs_noisy[i][0] = migrated_noisy[lbl] - migrated_noisy['Baseline']
    # Gazdag: same pattern
    try:
        imgs_noisy[i][1] = migrated_gz_diff_noisy[lbl]
    except (KeyError, NameError):
        if 'Baseline' in migrated_gz_noisy and lbl in migrated_gz_noisy:
            imgs_noisy[i][1] = migrated_gz_noisy[lbl] - migrated_gz_noisy['Baseline']

# LSM (CGLS) has no noisy counterpart in this notebook — column stays N/A.

# Back-propagation — reproject bp_ez_diff_noisy (if the noisy back-prop runs
# were completed) onto the migration grid, same reprojection as the clean cell.
try:
    bp_ez_diff_noisy
    x_ax_bp = np.linspace(0, 4.0, 4000)
    y_ax_bp = np.linspace(0, 1.0, 1000)
    y_mig   = y_surface - z_img
    Yq, Xq  = np.meshgrid(y_mig, x_traces, indexing='ij')
    for i, lbl in enumerate(labels):
        if lbl not in bp_ez_diff_noisy:
            print(f'[{lbl}] bp_ez_diff_noisy missing — skipping')
            continue
        interp = RegularGridInterpolator(
            (y_ax_bp, x_ax_bp), bp_ez_diff_noisy[lbl],
            method='linear', bounds_error=False, fill_value=0.0
        )
        imgs_noisy[i][3] = interp((Yq, Xq))
        print(f'[{lbl}] noisy back-prop diff reprojected  shape={imgs_noisy[i][3].shape}')
except NameError:
    print('bp_ez_diff_noisy not available — back-propagation (noisy) column left as N/A '
          '(noisy back-propagation was not re-run for this lateral dataset; see thesis §6.4).')

def _marker_z_cmp_noisy(i, j):
    return bp_z_top if j == 3 else z_top

plot_method_comparison_grid(
    imgs_noisy, extent_mig, methods, labels,
    title=f'TimeLapse Migration Comparison (Noisy) — Signed Amplitude  |  f_c={f_c_GHz} GHz  |  aperture={ANGLE_AP}°',
    envelope=False,
    marker_x=lambda i, j: x_scatterer_1[i + 1],
    marker_z=_marker_z_cmp_noisy,
    xlim=xlim, ylim=ylim,
)
plt.show()


In [ ]:
# ── Normalised lateral PSF of noisy timelapse difference at scatterer depth ──
# Mirrors the clean-data PSF cell above, fed imgs_noisy instead of imgs.
# Blue  = signed amplitude (normalised to peak)
# Red   = Hilbert envelope
# Green dashed = timelapsed scatterer x-position
# Green dotted  = baseline scatterer x-position

def _psf_profile_noisy(i, j):
    img = imgs_noisy[i][j]
    if img is None:
        return None
    marker_z = bp_z_top if j == 3 else z_top
    iz = int(np.argmin(np.abs(z_img - marker_z)))
    profile = img[iz, :].copy()
    peak = np.max(np.abs(profile))
    if peak > 0:
        profile /= peak
    return profile

profiles_noisy = [[_psf_profile_noisy(i, j) for j in range(n_m)] for i in range(n_s)]
marker_positions_noisy = [[[x_scatterer_1[i + 1], x_scatterer_1[0]] for _ in range(n_m)]
                           for i in range(n_s)]

fig_psf_noisy, axes_psf_noisy = plot_psf_grid(
    profiles_noisy, x_traces, methods, labels, orientation='lateral',
    marker_positions=marker_positions_noisy, ylim=(-1.15, 1.4),
    title=('Normalised Lateral PSF (Noisy) — TimeLapse Difference at True Scatterer Depth\n'
           'blue = amplitude  |  red = Hilbert envelope  '
           '|  green dashed = x_s1 timelapsed  |  green dotted = x_s1 baseline'),
)
for i in range(n_s):
    for j in range(n_m):
        axes_psf_noisy[i, j].set_xlim(x_scatterer_1[i + 1] - x_win, x_scatterer_1[i + 1] + x_win)
axes_psf_noisy[0, 0].legend(fontsize=7, loc='upper right')
plt.show()
